# Tools

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why do parrots talk")
response.content

'<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re known for mimicking human speech, right? But why do they do that? Maybe it\'s related to their environment or their social behavior. Let me break it down.\n\nFirst, I remember that parrots are social animals. They live in flocks in the wild. So maybe talking is a way to communicate with each other. But how does that translate to talking to humans? Maybe when they\'re around humans, they mimic the sounds they hear, just like they would mimic other parrots. That makes sense because they learn from their surroundings.\n\nThen there\'s the aspect of mimicry. Some birds, like mynas and crows, also mimic sounds, but parrots are especially good at it. Why is that? I think it has to do with their anatomy. They have a specialized vocal organ called the syrinx, which allows them to produce a wide range of sounds. Humans have vocal cords, but parrots\' syrinx might be more flexi

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What is the weather like in mumbai?")

response

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Mumbai. I need to use the get_weather function. The function requires the location parameter. Mumbai is the location here. So I should call get_weather with location set to "Mumbai". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'prts7b9hf', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 154, 'total_tokens': 250, 'completion_time': 0.14812499, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.006377297, 'prompt_tokens_details': None, 'queue_time': 0.055273443, 'total_time': 0.154502287}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 

In [5]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Mumbai'}


## Tool Execution Loops

In [ ]:
# Step-1 Model generates tool calls

message = [{"role":"user", "content":"What is the weather in mumbai?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

In [7]:
message

[{'role': 'user', 'content': 'What is the weather in mumbai?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Mumbai. I need to use the get_weather function. The function requires the location parameter. Mumbai is the location here. So I should call get_weather with location set to "Mumbai". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'j6x0vejn3', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 153, 'total_tokens': 249, 'completion_time': 0.149060191, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.007223594, 'prompt_tokens_details': None, 'queue_time': 0.799735697, 'total_time': 0.156283785}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'fi

In [8]:
# Step-2 Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

message

[{'role': 'user', 'content': 'What is the weather in mumbai?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Mumbai. I need to use the get_weather function. The function requires the location parameter. Mumbai is the location here. So I should call get_weather with location set to "Mumbai". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'j6x0vejn3', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 153, 'total_tokens': 249, 'completion_time': 0.149060191, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.007223594, 'prompt_tokens_details': None, 'queue_time': 0.799735697, 'total_time': 0.156283785}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'fi

In [10]:
# Step-3 pass the result back to the model for response

final_response = model_with_tools.invoke(message)
print(final_response.text)

The weather in Mumbai is currently sunny. ☀️ Let me know if you need more details!


In [13]:
message[-1]

ToolMessage(content="It's sunny in Mumbai", name='get_weather', tool_call_id='j6x0vejn3')